# 🎬 StudioCut — Notebook VTT Parser (`clip-cli`)

This Google Colab notebook receives a WebVTT (`.vtt`) subtitle/caption file (including YouTube auto-generated transcripts with rolling windows and inline timestamps `<hh:mm:ss.mmm>`), cleans the text using **word-level suffix-prefix overlap deduplication**, displays manual verification output, and stores the resulting cleaned transcript into **Google Drive** under the `clip-cli` directory.

## 📁 Step 1: Mount Google Drive

Mount Google Drive to enable reading/writing directly from your Drive storage inside the `clip-cli` directory.

In [1]:
#@title Mount Google Drive { display-mode: "form" }
import os

IS_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IS_COLAB = True
    print("✅ Google Drive successfully mounted.")
except ImportError:
    print("ℹ️ Running outside Google Colab environment. Operating in local mode.")

Mounted at /content/drive
✅ Google Drive successfully mounted.


## ⚙️ Step 2: Configuration & Parameters

Configure input/output parameters using native Colab form controls.

In [2]:
#@title Configuration Parameters { display-mode: "form" }
vtt_file_path = "sample.vtt" #@param {type:"string"}
drive_output_folder = "/content/drive/MyDrive/clip-cli" #@param {type:"string"}
output_filename = "cleaned_transcript.txt" #@param {type:"string"}
preview_cues_count = 10 #@param {type:"integer"}

print(f"📌 VTT File Path: {vtt_file_path}")
print(f"📌 Drive Output Folder: {drive_output_folder}")
print(f"📌 Output Filename: {output_filename}")

📌 VTT File Path: sample.vtt
📌 Drive Output Folder: /content/drive/MyDrive/clip-cli
📌 Output Filename: cleaned_transcript.txt


## 📥 Step 3: Input VTT File Ingestion

Load the VTT file from the specified path, or trigger Colab file upload if the specified file is not found.

In [3]:
import os

vtt_content = ""
target_path = vtt_file_path.strip()

if os.path.exists(target_path):
    with open(target_path, 'r', encoding='utf-8') as f:
        vtt_content = f.read()
    print(f"✅ Successfully loaded '{target_path}' ({len(vtt_content)} chars).")
else:
    print(f"⚠️ File '{target_path}' not found.")
    if IS_COLAB:
        from google.colab import files
        print("📤 Please upload a .vtt file:")
        uploaded = files.upload()
        for fname, data in uploaded.items():
            if fname.lower().endswith('.vtt'):
                vtt_content = data.decode('utf-8', errors='ignore')
                vtt_file_path = fname
                print(f"✅ Loaded uploaded file: {fname}")
                break
    else:
        raise FileNotFoundError(f"Cannot find VTT file at path: {target_path}")

⚠️ File 'sample.vtt' not found.
📤 Please upload a .vtt file:


Saving captions (1).vtt to captions (1).vtt
✅ Loaded uploaded file: captions (1).vtt


## 🧹 Step 4: YouTube Rolling Auto-Caption Deduplication Algorithm

YouTube auto-generated transcripts use **scrolling buffer windows** where each new line repeats previously spoken words and appends newly recognized words with inline timestamps `<hh:mm:ss.mmm>`.

The deduplication algorithm:
1. Strips inline timestamps and HTML formatting tags (`<[^>]*>`).
2. Performs a **word-level suffix-prefix overlap search** against recently emitted transcript words.
3. Appends only newly introduced words for each cue line, effectively neutralizing YouTube's rolling window repetitions.

In [4]:
import re
import json

def normalize_word(word):
    """Normalizes word for comparison (lowercase, alphanumeric characters only)."""
    return re.sub(r'[^\w]', '', word).lower()

def clean_cue_raw_text(raw_text):
    """Strips HTML tags and inline word timestamps e.g. <00:00:01.200>"""
    text = re.sub(r'<[^>]*>', '', raw_text)
    return re.sub(r'\s+', ' ', text).strip()

def clean_vtt_transcript(vtt_text):
    lines = vtt_text.splitlines()
    cues = []
    current_cue = None
    timestamp_pattern = re.compile(r'(\d{2}:)?\d{2}:\d{2}[\.,]\d{3}\s*-->\s*(\d{2}:)?\d{2}:\d{2}[\.,]\d{3}')

    for line in lines:
        line_str = line.strip()
        if not line_str or line_str.startswith('WEBVTT') or line_str.startswith('NOTE') or line_str.startswith('STYLE'):
            continue

        if timestamp_pattern.search(line_str):
            parts = line_str.split('-->')
            start = parts[0].strip()
            end = parts[1].strip().split()[0]
            current_cue = {'start': start, 'end': end, 'raw': [], 'cue_clean': '', 'incremental_clean': ''}
            cues.append(current_cue)
        elif current_cue is not None and not line_str.isdigit():
            current_cue['raw'].append(line_str)

    cumulative_words = []

    for cue in cues:
        raw_text = ' '.join(cue['raw'])
        cue_clean = clean_cue_raw_text(raw_text)
        cue['cue_clean'] = cue_clean

        if not cue_clean:
            cue['incremental_clean'] = ''
            continue

        words = cue_clean.split()
        max_overlap = 0
        max_lookback = min(len(cumulative_words), 50)

        # 1. Word-level suffix-prefix overlap matching
        for k in range(1, len(words) + 1):
            prefix = [normalize_word(w) for w in words[:k]]
            for start in range(len(cumulative_words) - max_lookback, len(cumulative_words) - k + 1):
                if start < 0: continue
                suffix = [normalize_word(w) for w in cumulative_words[start:start+k]]
                if prefix == suffix and (start + k == len(cumulative_words)):
                    if k > max_overlap:
                        max_overlap = k

        # 2. Sub-phrase match if cue is completely repeated line inside recent cumulative words
        if max_overlap == 0 and len(cumulative_words) > 0:
            norm_words = [normalize_word(w) for w in words]
            norm_cum = [normalize_word(w) for w in cumulative_words[-max_lookback:]]
            for i in range(len(norm_cum) - len(norm_words) + 1):
                if norm_cum[i:i+len(norm_words)] == norm_words:
                    max_overlap = len(words)
                    break

        new_words = words[max_overlap:]
        cumulative_words.extend(new_words)
        cue['incremental_clean'] = ' '.join(new_words)

    final_cleaned_transcript = ' '.join(cumulative_words)

    return {
        'cleaned_transcript': final_cleaned_transcript,
        'cues': cues,
        'total_cues': len(cues)
    }

parsing_result = clean_vtt_transcript(vtt_content)
print(f"✅ Cleaning complete. Parsed {parsing_result['total_cues']} cues.")
print(f"📊 Cleaned text length: {len(parsing_result['cleaned_transcript'])} characters.")

✅ Cleaning complete. Parsed 15 cues.
📊 Cleaned text length: 284 characters.


## 🔍 Step 5: Manual Verification of Outcome

Compare raw VTT cue text (with inline timestamps), full cue clean text, and newly added non-duplicate incremental text.

In [5]:
from IPython.display import HTML, display

def display_manual_verification(result, limit=10):
    cues = result['cues'][:limit]

    html_content = f"""
    <h3>🔍 Side-by-Side Manual Verification (First {limit} Cues)</h3>
    <table border="1" style="border-collapse: collapse; width: 100%; font-family: sans-serif;">
      <thead>
        <tr style="background-color: #f2f2f2; text-align: left;">
          <th style="padding: 8px; width: 12%;">Timestamp</th>
          <th style="padding: 8px; width: 38%;">Raw VTT Cue Text</th>
          <th style="padding: 8px; width: 25%;">Cue Text (Tags Stripped)</th>
          <th style="padding: 8px; width: 25%;">Newly Added Text (Deduplicated)</th>
        </tr>
      </thead>
      <tbody>
    """

    for i, c in enumerate(cues):
        raw_str = ' '.join(c['raw'])
        bg_color = "#ffffff" if i % 2 == 0 else "#f9f9f9"
        inc_display = c['incremental_clean'] if c['incremental_clean'] else '<i style="color:#888;">[Duplicate - Skipped]</i>'
        html_content += f"""
        <tr style="background-color: {bg_color};">
          <td style="padding: 8px; font-size: 11px; font-family: monospace;">{c['start']} &rarr; {c['end']}</td>
          <td style="padding: 8px; font-size: 12px; color: #cc0000; font-family: monospace;">{raw_str}</td>
          <td style="padding: 8px; font-size: 12px; color: #555555;">{c['cue_clean']}</td>
          <td style="padding: 8px; font-size: 12px; color: #008800; font-weight: bold;">{inc_display}</td>
        </tr>
        """

    html_content += """
      </tbody>
    </table>
    """
    display(HTML(html_content))

display_manual_verification(parsing_result, limit=preview_cues_count)

print("\n--- 📜 CLEANED TRANSCRIPT FULL PREVIEW ---")
print(parsing_result['cleaned_transcript'])
print("------------------------------------------\n")

Timestamp,Raw VTT Cue Text,Cue Text (Tags Stripped),Newly Added Text (Deduplicated)
00:00:00.160 → 00:00:02.270,"Cuando<00:00:00.799> una<00:00:01.040> pareja<00:00:01.439> dice,<00:00:01.760> ""Nunca<00:00:02.040> hemos","Cuando una pareja dice, ""Nunca hemos","Cuando una pareja dice, ""Nunca hemos"
00:00:02.270 → 00:00:02.280,"Cuando una pareja dice, ""Nunca hemos","Cuando una pareja dice, ""Nunca hemos",[Duplicate - Skipped]
00:00:02.280 → 00:00:03.909,"Cuando una pareja dice, ""Nunca hemos tenido<00:00:02.560> un<00:00:02.760> problema.""<00:00:03.480> O<00:00:03.560> se<00:00:03.719> están","Cuando una pareja dice, ""Nunca hemos tenido un problema."" O se están","tenido un problema."" O se están"
00:00:03.909 → 00:00:03.919,"tenido un problema."" O se están","tenido un problema."" O se están",[Duplicate - Skipped]
00:00:03.919 → 00:00:06.269,"tenido un problema."" O se están engañando<00:00:04.640> o<00:00:04.839> están<00:00:05.040> mintiendo.<00:00:05.759> Son<00:00:05.960> dos","tenido un problema."" O se están engañando o están mintiendo. Son dos",engañando o están mintiendo. Son dos
00:00:06.269 → 00:00:06.279,engañando o están mintiendo. Son dos,engañando o están mintiendo. Son dos,[Duplicate - Skipped]
00:00:06.279 → 00:00:08.070,"engañando o están mintiendo. Son dos personas<00:00:06.680> con<00:00:06.879> educaciones<00:00:07.439> diferentes,<00:00:07.919> con","engañando o están mintiendo. Son dos personas con educaciones diferentes, con","personas con educaciones diferentes, con"
00:00:08.070 → 00:00:08.080,"personas con educaciones diferentes, con","personas con educaciones diferentes, con",[Duplicate - Skipped]
00:00:08.080 → 00:00:10.150,"personas con educaciones diferentes, con valores<00:00:08.440> diferentes,<00:00:09.480> probablemente<00:00:09.920> hasta","personas con educaciones diferentes, con valores diferentes, probablemente hasta","valores diferentes, probablemente hasta"
00:00:10.150 → 00:00:10.160,"valores diferentes, probablemente hasta","valores diferentes, probablemente hasta",[Duplicate - Skipped]



--- 📜 CLEANED TRANSCRIPT FULL PREVIEW ---
Cuando una pareja dice, "Nunca hemos tenido un problema." O se están engañando o están mintiendo. Son dos personas con educaciones diferentes, con valores diferentes, probablemente hasta con proyectos diferentes. Tiene que haber dificultades, es la manera en que nosotros crecemos. M.
------------------------------------------



## 💾 Step 6: Export Output to Google Drive (`clip-cli`)

Save the cleaned transcript text (`.txt`) and parsed JSON metadata (`.json`) into Google Drive inside the `clip-cli` folder.

In [6]:
import os
import json

if IS_COLAB:
    target_dir = drive_output_folder
else:
    target_dir = "./output/clip-cli"

os.makedirs(target_dir, exist_ok=True)

output_txt_path = os.path.join(target_dir, output_filename)
with open(output_txt_path, 'w', encoding='utf-8') as f:
    f.write(parsing_result['cleaned_transcript'])

json_filename = os.path.splitext(output_filename)[0] + ".json"
output_json_path = os.path.join(target_dir, json_filename)
export_data = {
    "vtt_source": vtt_file_path,
    "total_cues": parsing_result['total_cues'],
    "cleaned_transcript": parsing_result['cleaned_transcript'],
    "cues": parsing_result['cues']
}
with open(output_json_path, 'w', encoding='utf-8') as f:
    json.dump(export_data, f, indent=2)

print(f"✅ Cleaned text file saved to: {output_txt_path}")
print(f"✅ Structured JSON file saved to: {output_json_path}")

✅ Cleaned text file saved to: /content/drive/MyDrive/clip-cli/cleaned_transcript.txt
✅ Structured JSON file saved to: /content/drive/MyDrive/clip-cli/cleaned_transcript.json
